# 03 · Face Swap Generation — InSwapper → CodeFormer → Real-ESRGAN

Generate synthetic face-swaps with InSwapper (`inswapper_128.onnx`). Refine each swap with CodeFormer (fidelity w=0.5) and upscale with Real-ESRGAN. Iterate over P50 and P99 pairs across all four source datasets.

In [ ]:
import os
import sys
import warnings
import logging
from contextlib import contextmanager
from io import StringIO

from telegram import send_message_telegram
import swapper as sw
import restoration as rt
import glob
from PIL import Image
import logging
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
warnings.filterwarnings('ignore')

@contextmanager
def suppress_output():
    """Suprime temporalmente stdout y stderr"""
    old_stdout = sys.stdout
    old_stderr = sys.stderr
    try:
        sys.stdout = StringIO()
        sys.stderr = StringIO()
        yield
    finally:
        sys.stdout = old_stdout
        sys.stderr = old_stderr


In [ ]:
def enhace(result_image):
    # rt.check_ckpts()
    
    
    upsampler = rt.set_realesrgan()
    device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
    
    codeformer_net = rt.ARCH_REGISTRY.get("CodeFormer")(dim_embd=512,
                                                     codebook_size=1024,
                                                     n_head=8,
                                                     n_layers=9,
                                                     connect_list=["32", "64", "128", "256"],
                                                    ).to(device)
    ckpt_path = "CodeFormer/CodeFormer/weights/CodeFormer/codeformer.pth"
    checkpoint = torch.load(ckpt_path)["params_ema"]
    codeformer_net.load_state_dict(checkpoint)
    codeformer_net.eval()
    
    result_image = cv2.cvtColor(np.array(result_image), cv2.COLOR_RGB2BGR)
    result_image = rt.face_restoration(result_image, 
                                    background_enhance = False, 
                                    face_upsample = True, 
                                    upscale = 1, 
                                    codeformer_fidelity = 0.5,
                                    upsampler = upsampler,
                                    codeformer_net = codeformer_net,
                                    device = device)
    result_image = Image.fromarray(result_image)
    return result_image
    

In [ ]:
# source_img = [Image.open(glob.glob('../flickr_cropped_and_aligned/*')[16])]
# target_img = Image.open(glob.glob('../flickr_cropped_and_aligned/*')[906])
source = 'claudia'
target = 'polo'
source_img = [Image.open(f'./{source}.png')]
target_img = Image.open(f'./{target}.png')

In [ ]:
model = "./checkpoints/inswapper_128.onnx"
with suppress_output():
    result_image = sw.process(source_img, target_img, -1, -1, model)
    img_enhaced = enhace(result_image)
# img_enhaced.save('./final.png');

In [ ]:
imagenes = [source_img[0], target_img, img_enhaced]
titulos = ['Source', 'Target', 'FaceSwap']

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, img, titulo in zip(axes, imagenes, titulos):
    ax.imshow(img)
    ax.set_title(titulo, fontsize=12)
    ax.axis("off")

plt.tight_layout()
# plt.savefig(f'./ejemplos/{source}_{target}_2.png', dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
flickr_swap_dataset = pd.read_parquet('../../00 Prepare Dataset/datasets/05 Curated/flickr_swap_dataset.parquet')
flickr_swap_dataset.head(1)

In [ ]:
train = flickr_swap_dataset[flickr_swap_dataset['split'] == 'train']
val = flickr_swap_dataset[flickr_swap_dataset['split'] == 'val']
test = flickr_swap_dataset[flickr_swap_dataset['split'] == 'test']
test

In [ ]:
model = "./checkpoints/inswapper_128.onnx"
contador = 0
for record in tqdm(test.to_dict(orient = 'records')):

    source_img_path = record['source']
    target_p50_img_path = record['target_p50']
    target_p99_img_path = record['target_p99']

    source_img_name = source_img_path.split('.')[0]
    target_p50_img_name = target_p50_img_path.split('.')[0]
    target_p99_img_name = target_p99_img_path.split('.')[0]
    
    source = [Image.open(f'../00 Images/flickr_cropped_and_aligned/{source_img_path}')]
    target_p50 = Image.open(f'../00 Images/flickr_cropped_and_aligned/{target_p50_img_path}')
    target_p99 = Image.open(f'../00 Images/flickr_cropped_and_aligned/{target_p99_img_path}')

    if os.path.exists(f'../01 Swaps/flickr/test/{source_img_name}_{target_p50_img_name}.png') and os.path.exists(f'../01 Swaps/flickr/test/{source_img_name}_{target_p99_img_name}.png'):
        continue
    with suppress_output():
        result_image_p50 = sw.process(source, target_p50, '-1', '-1', model)
        result_image_p99 = sw.process(source, target_p99, '-1', '-1', model)
        
        img_enhaced_p50 = enhace(result_image_p50)
        img_enhaced_p99 = enhace(result_image_p99)
        
        
        img_enhaced_p50.save(f'../01 Swaps/flickr/test/{source_img_name}_{target_p50_img_name}.png')
        img_enhaced_p99.save(f'../01 Swaps/flickr/test/{source_img_name}_{target_p99_img_name}.png')
        contador +=1
    
    if contador % 100 == 0:
        send_message_telegram(f'Iteración: {contador}')